# INFOSYS 722 Iteration 4 – BDAS Diabetes Prediction

Environment:
- AWS EC2
- Apache Spark
- PySpark
- Spark MLlib

Pipeline:
1. Data Preparation
2. Data Transformation
3. Model Selection and Data Mining
4. Interpretation and Evaluation

This notebook executes the final reproducible workflow.


In [1]:
import sys
import os

sys.path.insert(0, "src")

from spark_session import get_spark

spark = get_spark()

print("==============================")
print("ITERATION 4 BDAS START")
print("==============================")

print("Spark Version:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/ubuntu/.local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/05 07:41:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


ITERATION 4 BDAS START
Spark Version: 4.2.0


## Step 3 – Data Preparation

Operations:
- Remove invalid target records
- Remove duplicate response profiles
- Create missingness indicators
- Construct BMI bands, health bands, RiskTest5 and ComorbidityCount
- Create deterministic train/test partition


In [2]:
%run src/spark_preparation.py


STEP 3 - DATA PREPARATION



1. Original Dataset


Rows : 433323
Cols : 21
root
 |-- ID: integer (nullable = true)
 |-- Diabetes_binary: integer (nullable = true)
 |-- HighBP: integer (nullable = true)
 |-- HighChol: integer (nullable = true)
 |-- CholCheck: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker: integer (nullable = true)
 |-- Stroke: integer (nullable = true)
 |-- HeartDiseaseorAttack: integer (nullable = true)
 |-- PhysActivity: integer (nullable = true)
 |-- HvyAlcoholConsump: integer (nullable = true)
 |-- AnyHealthcare: integer (nullable = true)
 |-- NoDocbcCost: integer (nullable = true)
 |-- GenHlth: integer (nullable = true)
 |-- MentHlth: integer (nullable = true)
 |-- PhysHlth: integer (nullable = true)
 |-- DiffWalk: integer (nullable = true)
 |-- Sex: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Education: integer (nullable = true)
 |-- Income: integer (nullable = true)


ID removed



2. Target Cleaning
Removed: 984
Remaining: 432339



3. Duplicate Removal
Removed: 7391
Remaining: 424948


...

Rows: 424948
Columns: 33

Partition Distribution


+---------+------+
|Partition| count|
+---------+------+
| Training|297415|
|  Testing|127533|
+---------+------+


Target Distribution


26/10/05 07:41:50 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------------+------+
|Diabetes_binary| count|
+---------------+------+
|              1| 70152|
|              0|354796|
+---------------+------+


Missing Indicator Columns
Income_missing
HighChol_missing
BMI_missing
HvyAlcoholConsump_missing
CholCheck_missing
n_missing_fields

Final Schema
root
 |-- Diabetes_binary: integer (nullable = true)
 |-- HighBP: integer (nullable = true)
 |-- HighChol: integer (nullable = true)
 |-- CholCheck: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker: integer (nullable = true)
 |-- Stroke: integer (nullable = true)
 |-- HeartDiseaseorAttack: integer (nullable = true)
 |-- PhysActivity: integer (nullable = true)
 |-- HvyAlcoholConsump: integer (nullable = true)
 |-- AnyHealthcare: integer (nullable = true)
 |-- NoDocbcCost: integer (nullable = true)
 |-- GenHlth: integer (nullable = true)
 |-- MentHlth: integer (nullable = true)
 |-- PhysHlth: integer (nullable = true)
 |-- DiffWalk: integer (nullable = true)
 |-- Sex: inte


Saved:
data/clean_diabetes_step3.parquet


## Step 4 – Data Transformation

Operations:
- Training-only median imputation
- Feature vector assembly
- Chi-square feature selection
- StandardScaler projection

Output:
- train_step4_transformed.parquet
- test_step4_transformed.parquet


In [3]:
%run src/spark_transformation.py


STEP 4 - DATA TRANSFORMATION

1. Loaded Step 3 Output


Rows : 424948
Cols : 33

2. Partition Check


Training rows : 297415
Testing rows  : 127533

3. Input Features Before Reduction
Feature count : 31
HighBP
HighChol
CholCheck
BMI
Smoker
Stroke
HeartDiseaseorAttack
PhysActivity
HvyAlcoholConsump
AnyHealthcare
NoDocbcCost
GenHlth
MentHlth
PhysHlth
DiffWalk
Sex
Age
Education
Income
Income_missing
HighChol_missing
BMI_missing
HvyAlcoholConsump_missing
CholCheck_missing
n_missing_fields
BMI_band
MentHlth_band
PhysHlth_band
RiskTest5
RiskTest5_flag
ComorbidityCount



4. Imputation Completed
Training-only imputation model fitted and applied.

5. Raw Feature Vector Created
Vector column: raw_features



6. Feature Reduction Completed
Original feature count : 31
Selected feature count : 24

Selected Feature Indices:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 16, 17, 18, 20, 23, 24, 25, 26, 27, 28]

Selected Feature Names:
HighBP
HighChol
CholCheck
BMI
Smoker
Stroke
HeartDiseaseorAttack
PhysActivity
HvyAlcoholConsump
AnyHealthcare
GenHlth
MentHlth
PhysHlth
DiffWalk
Age
Education
Income
HighChol_missing
CholCheck_missing
n_missing_fields
BMI_band
MentHlth_band
PhysHlth_band
RiskTest5



7. Projection / Scaling Completed
Output vector column: features

FINAL STEP 4 AUDIT

Training rows : 297415


Testing rows  : 127533

Columns in transformed training set:
Diabetes_binary
HighBP
HighChol
CholCheck
BMI
Smoker
Stroke
HeartDiseaseorAttack
PhysActivity
HvyAlcoholConsump
AnyHealthcare
NoDocbcCost
GenHlth
MentHlth
PhysHlth
DiffWalk
Sex
Age
Education
Income
Income_missing
HighChol_missing
BMI_missing
HvyAlcoholConsump_missing
CholCheck_missing
n_missing_fields
BMI_band
MentHlth_band
PhysHlth_band
RiskTest5
RiskTest5_flag
ComorbidityCount
Partition
raw_features
selected_raw_features
features

Selected feature count: 24
Selected features:
- HighBP
- HighChol
- CholCheck
- BMI
- Smoker
- Stroke
- HeartDiseaseorAttack
- PhysActivity
- HvyAlcoholConsump
- AnyHealthcare
- GenHlth
- MentHlth
- PhysHlth
- DiffWalk
- Age
- Education
- Income
- HighChol_missing
- CholCheck_missing
- n_missing_fields
- BMI_band
- MentHlth_band
- PhysHlth_band
- RiskTest5



Saved outputs:
data/train_step4_transformed.parquet
data/test_step4_transformed.parquet
data/selected_features_step4.txt


## Steps 5–7 – Data Mining

Models:
- Logistic Regression
- Decision Tree (maxDepth=4)
- Random Forest
- Gradient Boosted Trees

Evaluation data remains the untouched testing partition.


In [4]:
%run src/spark_model.py


STEP 5 / 6 / 7 - METHODS, ALGORITHMS, DATA MINING

1. Loaded Step 4 Outputs


Training rows : 297415
Testing rows  : 127533



2. Class Weights
Negative class count : 248400
Positive class count : 49015
Negative class weight: 0.598661
Positive class weight: 3.033918

3. Baseline Evaluation


26/10/05 07:42:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 07:42:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 07:42:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 07:42:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/05 07:42:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 07:42:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 07:42:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 07:42:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 07:42:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


...

k = 4, silhouette = 0.166175


k = 5, silhouette = 0.192041


k = 6, silhouette = 0.171482

Best k selected: 5
Best silhouette: 0.192041

6. Final KMeans Fit and Cluster Profiling


+-------+------+-----------------+------------------+-----------------+------------------+------------------+---------------------+-------------------+
|cluster|n     |Age_mean         |GenHlth_mean      |Income_mean      |BMI_mean          |RiskTest5_mean    |ComorbidityCount_mean|Positive_Prevalence|
+-------+------+-----------------+------------------+-----------------+------------------+------------------+---------------------+-------------------+
|0      |43864 |9.036020426773664|3.9109748312967354|5.640662046325004|30.78528839139145 |2.8797419296005837|0.9877804121831114   |0.3475059274120007 |
|1      |115052|6.490743316065779|2.1467771094809303|7.720187393526405|26.457130862565776|1.1460904634426172|0.04534471369467719  |0.05889510829885617|
|2      |18087 |5.46668878199812 |2.6575440924420857|6.551722231436944|27.088730027091717|1.501299275722895 |0.20904517056449384  |0.0901199756731354 |
|3      |95015 |9.894490343629952|2.6713361048255537|6.8705257064674  |30.24634815555337

Saved: data/kmeans_cluster_profile.csv
Saved: data/kmeans_screening_results.csv

FINAL SUMMARY

Classification outputs saved:
- data/predictions_LogisticRegression.parquet
- data/predictions_DecisionTree.parquet
- data/predictions_RandomForest.parquet
- data/predictions_GBT.parquet
- data/model_metrics_step7.csv

Clustering outputs saved:
- data/kmeans_screening_results.csv
- data/kmeans_cluster_profile.csv

Done.


## Step 8 – Interpretation and Evaluation

Evaluation:
- AUC
- Positive-class precision and recall
- F1 score
- Confusion matrix
- Top-20 gain and lift
- 60% capture operating depth
- RiskTest5 baseline comparison
- Income-band equity analysis
- KMeans silhouette evaluation


In [5]:
%run src/spark_evaluation.py


STEP 8 - INTERPRETATION AND EVALUATION


Logistic Regression


/home/ubuntu/.local/lib/python3.12/site-packages/pyspark/sql/udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/ubuntu/.local/lib/python3.12/site-packages/pyspark/sql/udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(



Confusion Matrix
+---------------+----------+-----+
|Diabetes_binary|prediction|count|
+---------------+----------+-----+
|              1|       0.0| 5260|
|              0|       0.0|77271|
|              1|       1.0|15845|
|              0|       1.0|31280|
+---------------+----------+-----+




Metrics
AUC: 0.8044791780899182
Accuracy: 0.718177330782995
Precision: 0.3362334217506631
Recall: 0.7507699597251836
F1: 0.46445844936245057


26/10/05 07:46:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 07:46:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


...

+------------------+------------------+-----------------+-------------------+-------------------+-------------------+--------------------+-------------------+------------------+-------------------+------------------+
|AUC               |Accuracy          |Contacts_to_60pct|Depth_to_60pct     |F1                 |Model              |Places_Per_True_Case|Precision          |Recall            |Top20_Gain         |Top20_Lift        |
+------------------+------------------+-----------------+-------------------+-------------------+-------------------+--------------------+-------------------+------------------+-------------------+------------------+
|0.8044791780899182|0.718177330782995 |32558            |0.25111063120873695|0.46445844936245057|Logistic Regression|2.3912762818148283  |0.3362334217506631 |0.7507699597251836|0.5138118929163705 |2.5690594645818523|
|0.7055394415215654|0.688259702597643 |36272            |0.27975566113407785|0.44760902543357334|Decision Tree      |2.5794290261613

26/10/05 07:46:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 07:46:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 07:46:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/05 07:46:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Baseline Top20 Capture: 0.40568585643212507
Baseline Places per True Case: 3.0286148096239196


BO4 EQUITY ANALYSIS


+------+------+---------------+-----------+--------------------+--------------------+--------------------+
|Income|GBT_TP|Actual_Positive|RiskTest_TP|          GBT_Recall|     RiskTest_Recall|          Difference|
+------+------+---------------+-----------+--------------------+--------------------+--------------------+
|     0|   697|           4181|        260| 0.16670652953838794| 0.06218607988519493| 0.10452044965319301|
|     1|   188|            608|         49|  0.3092105263157895|  0.0805921052631579| 0.22861842105263158|
|     2|   306|            907|         90| 0.33737596471885334| 0.09922822491730982| 0.23814773980154352|
|     3|   317|            995|        105|  0.3185929648241206| 0.10552763819095477|  0.2130653266331658|
|     4|   396|           1388|        140| 0.28530259365994237| 0.10086455331412104| 0.18443804034582134|
|     5|   591|           2491|        204| 0.23725411481332798| 0.08189482135688479| 0.15535929345644317|
|     6|   507|           2670|      

Silhouette: 0.15494829834321938
+----------+------+
|prediction| count|
+----------+------+
|         1| 88024|
|         3| 49919|
|         4| 40129|
|         2| 12759|
|         0|111852|
+----------+------+


STEP 8 COMPLETED


In [6]:
import os

required_files = [
    "data/clean_diabetes_step3.parquet",
    "data/train_step4_transformed.parquet",
    "data/test_step4_transformed.parquet",
    "data/result_logistic.parquet",
    "data/result_tree.parquet",
    "data/result_rf.parquet",
    "data/result_gbt.parquet",
    "data/result_cluster.parquet"
]

print("==============================")
print("FINAL OUTPUT AUDIT")
print("==============================")

for file in required_files:
    if os.path.exists(file):
        print("FOUND:", file)
    else:
        print("MISSING:", file)

print("==============================")
print("ITERATION 4 COMPLETED")
print("==============================")


FINAL OUTPUT AUDIT
FOUND: data/clean_diabetes_step3.parquet
FOUND: data/train_step4_transformed.parquet
FOUND: data/test_step4_transformed.parquet
FOUND: data/result_logistic.parquet
FOUND: data/result_tree.parquet
FOUND: data/result_rf.parquet
FOUND: data/result_gbt.parquet
FOUND: data/result_cluster.parquet
ITERATION 4 COMPLETED
